# Drilling Activity Prediction: Data Modelling - ARIMA

## What is Arima:

**Autoregressive Integrated Moving Average (ARIMA)** are a class of statisical models used to analyze time-series data for analysis and forecasting. ARIMA strips away non-stationary elements via differencing, capturing correlation with past values using autoregression, and utilizes moving averages of past errors to model sudden system shocks. This is all meant to determine temporal structures (dependencies, trends and shocks); evident in historical values to predict fruture values

ARIMA is dependent on three parameters: `ARIMA(p,d,q)`. Each one encapsulates a mathematical operation to identify properties of time-series data.

### 1. Integrated Component $d$ (Differencing)

Real world data (such as stock prices) often show an inflation factor - a trend over time that is external to the component's data. Because of this, data tends to drift (**Data Drift**) and is more difficult to model. We can remove this factor by making the time-series data **Non-stationary**, removing external trends over time - thus we can isolate the inherent true trend of the data. 

Stationary data is evident by it's constant statistical properties - mean, variance and autocorrelation. The integrated (`d`) component converts non-stationary data to stationary by taking the differences of ith data point to `d`. 

**First Order Differencing** is the difference of data points against time as: 

$y'_{t} = y_t - y_{t-1}$

**Second Order Differencing** is the difference of the differences (first order) of the points:

$y''_t = y'_t - y'_{t-1} = (yt-y_{t-1}) - (y_{t-1} - y_{t-2}) = y_t = 2y_{t-1} + y_{t-2}$

### 2. Autoregressive Component: $p$ (Lagged Values)

This is essentially the look back value -> how many units in the time-scale are you going to look back. However, this assumes that staitonary series $(y'_t)$ depends linearly on its own past (lagged) values. You can set the degree of look back with `p`. 

$AR(p)$ is modelled as
$y'_t = c + \phi_{1y'_{t-1}} + \phi_{2y'_{t-2}} + ... + \phi_{p2'_{t-p}} + \epsilon_t$

Where: 
* $c$ is the constant term (intercept)
* $\phi_1 ... $\phi_p$ are the coefficients to be estimated
* \epsilon_t is white noise (random error at time t)

### 3. Moving Average Component: $q$ (Lagged Errors)

Measures the residual deviations from the mean -> which models the current value of the series as a linear combination of past forecast errors (shocks). 

$y'_t = \mu + \epsilon_t + \phi_1\epsilon_{t-1} + \phi_2epsilon_{t-2} + ... + \phi_q\epsilon_{t-q}$

Where:
* $\mu$ is the expectation of the series
* $\phi_1, ... \phi_q$ are the error wieght parameters to be estimated
* $\epsilon_{t-1},...\epsilon_{t-q}$ are the historical residuals (actual value minus predicted value).

### Combined ARIMA Equation

Mathematical equation for a stationary series modelled by $ARIMA(p,q,d)$ is expressed using backshift notation $(By_t = y_{t-1})$ as: 

$(1- \sum_{i=1}^{p}\phi_iB^i)(1-B)^dy_t = c + (1+\sum_{j=1}^{q}\phi_jB^j)\epsilon_t$

In [33]:
import duckdb
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_colwidth', None)

# Load specific forecasting tools
from statsmodels.tsa.arima.model import ARIMA,ARIMAResults
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tools.eval_measures import rmse, mse, meanabs
from sklearn.model_selection import TimeSeriesSplit
import time
from pmdarima import auto_arima
from pylab import rcParams

# Ignore harmless warnings
import warnings
warnings.filterwarnings("ignore")

# Size of all plots
rcParams['figure.figsize'] = 6,6

In [5]:
conn = duckdb.connect()

# DuckDB connection to PostgreSQL
conn.execute("INSTALL postgres")
conn.execute("LOAD postgres")

# Attach to PostgreSQL database directly
conn.execute(f"""
    ATTACH '
            host={os.getenv("POSTGRES_HOST")} 
            port={os.getenv("POSTGRES_PORT", 5432)} 
            dbname={os.getenv("POSTGRES_DB")}
            user={os.getenv("POSTGRES_USER")}
            password={os.getenv("POSTGRES_PASSWORD")}
            '
    AS pg (TYPE postgres)
""")


In [8]:
ml_oil_set = conn.sql("""
    SELECT * FROM pg.silver.t_oil_ml_set
""")

ml_oil_set = ml_oil_set.to_df().set_index("period")


In [9]:
ml_gas_set = conn.sql("""
    SELECT * FROM pg.silver.t_gas_ml_set
""")

ml_gas_set = ml_gas_set.to_df().set_index("period")

In [ ]:
# Number of observations
nobs = 12

## Data Fitting

### Oil Rig Count Modelling

In [34]:
# What are the attributes of the dataset?
series_mapping_oil = conn.sql(f"""
    SELECT 
        LOWER(msn) AS msn
        ,series_description
        ,unit
    FROM pg.silver.v_total_energy_series
    WHERE msn IN ({", ".join([f"'{col.upper()}'" for col in ml_oil_set.columns])})
    """)

display(series_mapping_oil.df())

,msn,series_description,unit
0,rfwhuus,"Average Refiner Price of Residual Fuel Oil, Sales for Resale in Dollars per Gallon Excluding Taxes",Dollars per Gallon Excluding Taxes
1,d2tcuus,Average Refiner Price of No. 2 Fuel Oil to End Users in Dollars per Gallon Excluding Taxes,Dollars per Gallon Excluding Taxes
2,rfaceus,Residual Fuel Oil Transportation Sector CO2 Emissions in Million Metric Tons of Carbon Dioxide,Million Metric Tons of Carbon Dioxide
3,dmtceus,"Distillate Fuel Oil, Excluding Biodiesel, CO2 Emissions in Million Metric Tons of Carbon Dioxide",Million Metric Tons of Carbon Dioxide
4,jfaceus,Jet Fuel Transportation Sector CO2 Emissions in Million Metric Tons of Carbon Dioxide,Million Metric Tons of Carbon Dioxide
5,mmcceus,"Motor Gasoline, Excluding Ethanol, Commercial Sector CO2 Emissions in Million Metric Tons of Carbon Dioxide",Million Metric Tons of Carbon Dioxide
6,avtceus,Aviation Gasoline CO2 Emissions in Million Metric Tons of Carbon Dioxide,Million Metric Tons of Carbon Dioxide
7,rbtcuus,"Average Refiner Price of Residual Fuel Oil, Sulfur Content Less Than or Equal to 1 Percent, Sales to End Users in Dollars per Gallon Excluding Taxes",Dollars per Gallon Excluding Taxes
8,ngaceus,Natural Gas Transportation Sector CO2 Emissions in Million Metric Tons of Carbon Dioxide,Million Metric Tons of Carbon Dioxide
9,rfl1pus,"Residual Fuel Oil Consumption for Electricity Generation, Electric Power Sector in Thousand Barrels",Thousand Barrels


In [10]:
# Number of observations
nobs = 12 # For 12 months?

In [35]:
stepwise_fit = auto_arima(
    ml_oil_set[''], 
    start_p=0, 
    start_q=0,
    max_p=6,
    max_q=3,
    m=nobs,
    seasonal=True,
    d=None,
    trace=True,
    error_action='ignore',
    suppress_warnings=True,
    stepwise=True
    )

KeyError: ''